# Data Source 01 · 금융·투자

**목적:** 수집할 시계열의 단위·주기·결측·발표시점을 점검한다.

| 출처 | 접근 | 검증 상태 |
|---|---|---|
| FRED `fredgraph.csv` (예: `DGS10` 10년 국채금리) | 무키 | B (엔드포인트·헤더 수업 전 확인) |
| Yahoo Finance (`yfinance`) | 무키(비공식) | B |
| 한국은행 ECOS | 무료키 | C |
| Ken French Data Library | 수동 | C |

자세한 목록: `resources/Data_Source_Catalog.md`. **API 키는 파일에 쓰지 말고 `getpass`/환경변수로만 입력**한다.
온라인 연결이 실패하면 합성자료로 대체하며 `DATA_KIND`로 구분한다. 합성자료의 결과는 실제 현상의 증거가 아니다.

In [ ]:
import io, json, urllib.request, numpy as np, pandas as pd, matplotlib.pyplot as plt

def fetch_bytes(url, timeout=30):
    """무키 공개 URL 요청. 실패 시 예외 -> 호출부에서 합성 대체."""
    req = urllib.request.Request(url, headers={"User-Agent": "ts-lab-course/1.0 (education)"})
    with urllib.request.urlopen(req, timeout=timeout) as r:
        return r.read()

def synthetic_series(n, freq, seed, trend=0.05, season=12, noise=1.0, start="2015-01-01"):
    """NOT real data: 방법 시연용 합성 시계열."""
    rng = np.random.default_rng(seed)
    t = np.arange(n)
    y = 100 + trend * t + 3 * np.sin(2 * np.pi * t / season) + rng.normal(0, noise, n)
    return pd.Series(y, index=pd.date_range(start, periods=n, freq=freq), name="value")

def quality_report(s):
    s = s.sort_index()
    print("관측 수:", len(s), "| 기간:", s.index.min(), "~", s.index.max())
    print("결측:", int(s.isna().sum()), "| 중복 시점:", int(s.index.duplicated().sum()))
    f = pd.infer_freq(s.index) if len(s) > 2 else None
    print("추정 주기:", f)

In [ ]:
def parse_fred_csv(raw):
    """FRED fredgraph.csv 파싱. 날짜 열 이름이 DATE 또는 observation_date일 수 있고, 결측은 '.' 로 표시될 수 있다."""
    df = pd.read_csv(io.BytesIO(raw) if isinstance(raw, (bytes, bytearray)) else io.StringIO(raw))
    date_col = next(c for c in df.columns if c.lower() in ("date", "observation_date"))
    val_col = [c for c in df.columns if c != date_col][0]
    s = pd.Series(pd.to_numeric(df[val_col], errors="coerce").values,
                  index=pd.to_datetime(df[date_col]), name=val_col)
    return s

In [ ]:
SERIES_ID = "DGS10"   # FRED 시리즈 ID. 실제 존재·단위는 FRED 페이지에서 확인할 것.
try:
    raw = fetch_bytes(f"https://fred.stlouisfed.org/graph/fredgraph.csv?id={SERIES_ID}")
    y = parse_fred_csv(raw)
    DATA_KIND = "REAL: FRED " + SERIES_ID
except Exception as e:
    print("온라인 수집 실패 -> 합성 대체:", type(e).__name__, e)
    y = synthetic_series(1500, "B", 2026011, trend=0.002, season=250, noise=0.15, start="2018-01-01")
    DATA_KIND = "SYNTHETIC (NOT real data)"
print("DATA_KIND:", DATA_KIND)
quality_report(y)
y.plot(title=DATA_KIND); plt.show()

## 점검 체크리스트
- 단위(%, 지수, 수준)와 **주기**(영업일/월)가 연구질문과 맞는가
- 결측(휴장일)·대체 규칙을 사전에 정의했는가
- 수정주가/발표시점(빈티지): 사후 개정 값을 쓰면 look-ahead 위험
- 라이선스·재배포 조건: 원자료는 저장소에 올리지 않는다